# 11.4 — Bucketing Needs the Catalog

**Chapter 11, §3.3, layout decision 3 (lines 240–255), and Exercise 4.**

The chapter states the mechanism — records are assigned to $M$ buckets by
$\operatorname{hash}(k) \bmod M$, and two tables bucketed compatibly on the join key need less
redistribution — and then adds the condition that matters in practice: *"Without the
corresponding metadata, the presence of similarly organized files alone does not necessarily
provide the optimizer with enough information."*

This notebook writes the same physical layout twice, once registered in the catalog and once not,
and reads the difference off the query plan. It is the smallest demonstration that a physical
property nobody has recorded is not a property the optimizer can use.

In [1]:
import os, shutil, tempfile
from pyspark.sql import SparkSession, functions as F

SCRATCH = os.path.join(tempfile.gettempdir(), "cs777", "ch11-bucketing")
shutil.rmtree(SCRATCH, ignore_errors=True)
os.makedirs(SCRATCH)

spark = (
    SparkSession.builder.appName("CS777-11.4")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", 8)
    # Both switches are here so that the plans below show the join this notebook is about.
    # A broadcast join has no exchange either, and adaptive execution rewrites the plan after
    # the fact -- neither would let the bucketing be seen.
    .config("spark.sql.autoBroadcastJoinThreshold", -1)
    .config("spark.sql.adaptive.enabled", "false")
    .config("spark.ui.showConsoleProgress", "false")
    .config("spark.sql.warehouse.dir", os.path.join(SCRATCH, "warehouse"))
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
spark.version

'4.2.0'

In [2]:
orders = (spark.range(0, 200_000)
          .withColumn("customer_id", (F.col("id") % 2_000).cast("int"))
          .withColumn("amount", (F.col("id") % 97).cast("double"))
          .select(F.col("id").alias("order_id"), "customer_id", "amount"))

customers = (spark.range(0, 2_000)
             .withColumn("country", F.element_at(F.array(F.lit("US"), F.lit("DE"), F.lit("FR")),
                                                 (F.col("id") % 3 + 1).cast("int")))
             .select(F.col("id").cast("int").alias("customer_id"), "country"))

print(orders.count(), "orders /", customers.count(), "customers")

200000 orders / 2000 customers


## 1. The join without bucketing

A join on `customer_id` has to bring matching keys together on the same machine. With nothing
known about how the two inputs are laid out, that means a shuffle of both sides.

In [3]:
def exchanges(df, on=None):
    # The shuffle boundaries the planner inserted, optionally only those on one column.
    plan = df._jdf.queryExecution().executedPlan().toString()
    lines = [l.strip() for l in plan.split("\n") if "Exchange hashpartitioning" in l]
    if on is not None:
        lines = [l for l in lines if f"hashpartitioning({on}#" in l]
    return [l[:78] for l in lines]


plain = os.path.join(SCRATCH, "plain")
orders.write.mode("overwrite").parquet(os.path.join(plain, "orders"))
customers.write.mode("overwrite").parquet(os.path.join(plain, "customers"))

o = spark.read.parquet(os.path.join(plain, "orders"))
c = spark.read.parquet(os.path.join(plain, "customers"))
joined = o.join(c, "customer_id").groupBy("country").agg(F.sum("amount").alias("total"))

print("exchanges on customer_id:", len(exchanges(joined, "customer_id")))
for line in exchanges(joined):
    print(line)
joined.orderBy("country").show()

exchanges on customer_id: 2
+- Exchange hashpartitioning(country#24, 8), ENSURE_REQUIREMENTS, [plan_id=172
:  +- Exchange hashpartitioning(customer_id#21, 8), ENSURE_REQUIREMENTS, [plan
+- Exchange hashpartitioning(customer_id#23, 8), ENSURE_REQUIREMENTS, [plan_id


+-------+---------+
|country|    total|
+-------+---------+
|     DE|3201446.0|
|     FR|3196588.0|
|     US|3201385.0|
+-------+---------+



Two exchanges on `customer_id`, one per side, before the join can start. Every row of both tables
crosses the network once.

## 2. The same join over bucketed tables

`bucketBy` sends a row to file $\operatorname{hash}(\text{customer\_id}) \bmod 8$. If both tables
agree on the key and the number of buckets, the rows that have to meet are already in
corresponding files.

In [4]:
spark.sql("DROP TABLE IF EXISTS orders_bucketed")
spark.sql("DROP TABLE IF EXISTS customers_bucketed")

orders.write.mode("overwrite").bucketBy(8, "customer_id").sortBy("customer_id") \
      .saveAsTable("orders_bucketed")
customers.write.mode("overwrite").bucketBy(8, "customer_id").sortBy("customer_id") \
         .saveAsTable("customers_bucketed")

ob = spark.table("orders_bucketed")
cb = spark.table("customers_bucketed")
joined_b = ob.join(cb, "customer_id").groupBy("country").agg(F.sum("amount").alias("total"))

print("exchanges on customer_id:", len(exchanges(joined_b, "customer_id")))
for line in exchanges(joined_b):
    print(line)
joined_b.orderBy("country").show()

exchanges on customer_id: 0
+- Exchange hashpartitioning(country#48, 8), ENSURE_REQUIREMENTS, [plan_id=471


+-------+---------+
|country|    total|
+-------+---------+
|     DE|3201446.0|
|     FR|3196588.0|
|     US|3201385.0|
+-------+---------+



The same answer, and the shuffle on `customer_id` is gone: the only exchange left is the one the
`groupBy("country")` needs, which has nothing to do with the join key. That is the saving the
chapter describes — and it is repeated on every join of these two tables, not just this one.

## 3. The same bytes, without the catalog entry

Now the point of §3.3's last paragraph. `bucketBy` is not even offered to a path write:

In [5]:
from pyspark.errors import AnalysisException

try:
    orders.write.bucketBy(8, "customer_id").save(os.path.join(SCRATCH, "nope"))
except AnalysisException as exc:
    print(str(exc).split("\n")[0])

'save' does not support bucketBy right now.


So a writer who wants bucket-organized files at a path has to produce the layout by hand. This
does produce the layout: `repartition(8, "customer_id")` uses the same hash partitioner, so the
files contain exactly the bucket assignment `bucketBy` would have written.

In [6]:
byhand = os.path.join(SCRATCH, "byhand")
orders.repartition(8, "customer_id").write.mode("overwrite").parquet(os.path.join(byhand, "orders"))
customers.repartition(8, "customer_id").write.mode("overwrite").parquet(os.path.join(byhand, "customers"))

oh = spark.read.parquet(os.path.join(byhand, "orders"))
ch = spark.read.parquet(os.path.join(byhand, "customers"))
joined_h = oh.join(ch, "customer_id").groupBy("country").agg(F.sum("amount").alias("total"))

n = len([f for f in os.listdir(os.path.join(byhand, "orders")) if f.endswith(".parquet")])
print("files:", n, "in each table")
print("exchanges on customer_id:", len(exchanges(joined_h, "customer_id")))
for line in exchanges(joined_h):
    print(line)

files: 8 in each table
exchanges on customer_id: 2
+- Exchange hashpartitioning(country#72, 8), ENSURE_REQUIREMENTS, [plan_id=750
:  +- Exchange hashpartitioning(customer_id#69, 8), ENSURE_REQUIREMENTS, [plan
+- Exchange hashpartitioning(customer_id#71, 8), ENSURE_REQUIREMENTS, [plan_id


Eight files per table, partitioned by the same hash of the same key as the bucketed tables — and
both sides are shuffled anyway. The planner reads a directory of Parquet files; nothing in a
Parquet footer says "these rows are all the rows whose key hashes to 3", so the layout is
invisible and has to be re-established at query time.

What the catalog holds is exactly the missing sentence:

In [7]:
described = spark.sql("DESCRIBE EXTENDED orders_bucketed").collect()
for row in described:
    if row["col_name"] in ("Num Buckets", "Bucket Columns", "Sort Columns", "Provider", "Type"):
        print(f"{row['col_name']:<16} {row['data_type']}")

Type             MANAGED
Provider         parquet
Num Buckets      8
Bucket Columns   [`customer_id`]
Sort Columns     [`customer_id`]


`Num Buckets 8` and `Bucket Columns [customer_id]` are the two facts that turn a directory of
files into a distribution the optimizer can reason about. They live in the catalog, not in the
data — which is why §7 can say that the catalog participates in query planning and not only in
discovery.

## 4. What bucketing costs

Exercise 4(b): a table written into $M$ buckets by $T$ uncoalesced tasks can produce up to
$M \times T$ files, because every task contributes at most one file per bucket it holds a row
for.

In [8]:
loc = spark.sql("DESCRIBE EXTENDED orders_bucketed") \
           .filter("col_name = 'Location'").collect()[0]["data_type"]
path = loc.replace("file:", "")
n_files = len([f for f in os.listdir(path) if f.endswith(".parquet")])
print(f"{n_files} files for 8 buckets written by {spark.sparkContext.defaultParallelism} tasks")
print(f"Exercise 4(b): 96 buckets x 200 tasks = {96 * 200:,} files")

144 files for 8 buckets written by 18 tasks
Exercise 4(b): 96 buckets x 200 tasks = 19,200 files


The remedy is the one 11.3 used: shuffle the data onto the bucket key before writing, so each
bucket is held by one task and written as one file. The write pays a shuffle; every later join
on the key saves one.

## Conclusion

* Two tables bucketed on the join key with the same number of buckets join without redistributing
  either side. The exchange that remains in the plan belongs to the `groupBy`, not the join.
* The identical physical layout, written to a path instead of a table, buys nothing: both sides
  are shuffled again. Bucketing is a *claim about the data* that only counts once it is recorded,
  and `saveAsTable` is what records it.
* The layout is not free — up to $M \times T$ files on write, and a table that is only worth
  maintaining if the same key is joined repeatedly. Exercise 4(c) is asking for that condition:
  when the join is rare, or the key changes, or the tables are re-written more often than they
  are joined, the shuffle the bucketing avoids is cheaper to pay than to prevent.

In [9]:
spark.sql("DROP TABLE IF EXISTS orders_bucketed")
spark.sql("DROP TABLE IF EXISTS customers_bucketed")
spark.stop()